<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
《<a href="https://mng.bz/lZ5B">从零构建推理模型</a>》一书的配套代码，作者：<a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库： <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：练习解决方案

此笔记本中使用的库：

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # 由 reasoning_from_scratch 使用
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## 练习 2.1：对未知单词进行编码

In [ ]:
from pathlib import Path

from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
)

download_qwen3_small(kind="base", tokenizer_only=True, out_dir="qwen3")

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

prompt = "Hello, Ardwarklethyrx. Haus und Garten."
input_token_ids_list = tokenizer.encode(prompt)

for i in input_token_ids_list:
    print(f"{[i]} --> {tokenizer.decode([i])}")

- 未知单词会被拆分成更小的子词甚至单个 token，这样 tokenizer 和 LLM 就能处理任何输入。
- 德语单词（Haus und Garten）在这里没有被拆分，说明 tokenizer 训练时见过德语文本，模型本身也很可能包含德语训练数据。

&nbsp;
## 练习 2.2：在 GPU 设备上运行代码

- 只需删除第 2.5 节中的 `device = torch.device("cpu")` 这一行，然后重新运行代码

- 为方便起见，下面提供了一个来自第 2 章相关代码的最小自包含示例

In [ ]:
from pathlib import Path
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream,
    generate_text_basic_stream_cache,
    generate_stats
)
from reasoning_from_scratch.qwen3 import (
    download_qwen3_small,
    Qwen3Tokenizer,
    Qwen3Model,
    QWEN_CONFIG_06_B
)

device = get_device()
device = torch.device("cpu")

download_qwen3_small(kind="base", tokenizer_only=False, out_dir="qwen3")

tokenizer_path = Path("qwen3") / "tokenizer-base.json"
model_path = Path("qwen3") / "qwen3-0.6B-base.pth"

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(model_path))

model.to(device);

In [ ]:
prompt = "Explain large language models in 1 sentence."
input_token_ids_tensor = torch.tensor(
    tokenizer.encode(prompt),
    device=device
    ).unsqueeze(0)

In [ ]:
import time


max_new_tokens = 100
start_time = time.time()
generated_ids = []

for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的词元

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

In [ ]:
start_time = time.time()

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=max_new_tokens,
    eos_token_id=tokenizer.eos_token_id
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

    next_token_id = token.squeeze(0)
    generated_ids.append(next_token_id)  # 收集生成的词元

end_time = time.time()

output_token_ids_tensor = torch.cat(generated_ids, dim=0)
generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

In [ ]:
if device.type == "mps":
    print(f"`torch.compile` is not supported for the {model.__class__.__name__} model on MPS (Apple Silicon) as of this writing.")
    model_compiled = model
    # 进行赋值，避免使用“运行所有单元”时 notebook 在此停止
else:
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 8):
        # 这可以避免再次触发模型重新编译 
        # 适用于 PyTorch 2.8 及更高版本
        # 当模型包含 self.pos = self.pos + 1 之类代码时
        torch._dynamo.config.allow_unspec_int_on_nn_module = True
        
    model_compiled = torch.compile(model)

In [ ]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的词元
    
    end_time = time.time()
    

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

In [ ]:
for i in range(3):

    start_time = time.time()
    generated_ids = []
    
    for token in generate_text_basic_stream_cache(
        model=model_compiled,
        token_ids=input_token_ids_tensor,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id
    ):
        token_id = token.squeeze(0).tolist()
        print(
            tokenizer.decode(token_id),
            end="",
            flush=True
        )
    
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id)  # 收集生成的词元
    
    end_time = time.time()
    

    if i == 0:
        print("Warm-up run")
    else:
        print(f"Timed run {i}:")

    output_token_ids_tensor = torch.cat(generated_ids, dim=0)
    generate_stats(output_token_ids_tensor, tokenizer, start_time, end_time)

    print(f"\n{30*'-'}\n")

| 生成词元数  | 模式              | 硬件        | 词元/秒    | GPU 显存（VRAM） |
|-------------------|-------------------|-----------------|---------------|-------------------|
| 41                | 常规模式           | Mac Mini M4 CPU | 6             | -                 |
| 41                | 常规模式（已编译）  | Mac Mini M4 CPU | 6             | -                 |
| 41                | KV 缓存          | Mac Mini M4 CPU | 28            | -                 |
| 41                | KV 缓存（已编译） | Mac Mini M4 CPU | 68            | -                 |
|                   |                   |                 |               |                   |
| 41                | 常规模式           | Mac Mini M4 GPU | 17            | -                 |
| 41                | 常规模式（已编译）  | Mac Mini M4 GPU | InductorError | -                 |
| 41                | KV 缓存          | Mac Mini M4 GPU | 18            | -                 |
| 41                | KV 缓存（已编译） | Mac Mini M4 GPU | InductorError | -                 |
|                   |                   |                 |               |                   |
| 41                | 常规模式           | NVIDIA H100 GPU | 51            | 1.55 GB           |
| 41                | 常规模式（已编译）  | NVIDIA H100 GPU | 164           | 1.81 GB           |
| 41                | KV 缓存          | NVIDIA H100 GPU | 48            | 1.52 GB           |
| 41                | KV 缓存（已编译） | NVIDIA H100 GPU | 141           | 1.81 GB           |